In [1]:
# ============================================================
# CELL 1: Project configuration and environment validation
# ============================================================

from pathlib import Path
import sys
import os
import json
import shutil
import platform
import importlib.util
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError


# ------------------------------------------------------------
# 1. Define permanent project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

SCGPT_REPO = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "code/scGPT-main"
).resolve()

DIRS = {
    "project": PROJECT_DIR,
    "raw": PROJECT_DIR / "data" / "raw",
    "interim": PROJECT_DIR / "data" / "interim",
    "processed": PROJECT_DIR / "data" / "processed",
    "metadata": PROJECT_DIR / "data" / "metadata",
    "models": PROJECT_DIR / "models",
    "embeddings": PROJECT_DIR / "embeddings",
    "figures": PROJECT_DIR / "figures",
    "tables": PROJECT_DIR / "tables",
    "logs": PROJECT_DIR / "logs",
    "notebooks": PROJECT_DIR / "notebooks",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Define GEO accession and remote files
# ------------------------------------------------------------

GEO_ACCESSION = "GSE200164"

GEO_SERIES_URL = (
    "https://www.ncbi.nlm.nih.gov/geo/query/"
    f"acc.cgi?acc={GEO_ACCESSION}"
)

GEO_SUPPLEMENT_BASE = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE200nnn/GSE200164/suppl"
)

RAW_COUNTS_URL = (
    f"{GEO_SUPPLEMENT_BASE}/GSE200164_counts.csv.gz"
)

LOGNORM_URL = (
    f"{GEO_SUPPLEMENT_BASE}/"
    "GSE200164_lognorm_expression.csv.gz"
)

RAW_COUNTS_FILE = DIRS["raw"] / "GSE200164_counts.csv.gz"

# We are deliberately not downloading the log-normalized matrix.
# Normalization will be performed from the raw UMI counts.


# ------------------------------------------------------------
# 3. Validate the local scGPT repository
# ------------------------------------------------------------

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

for name, path in DIRS.items():
    print(f"{name:12s}: {path}")

print("\n" + "=" * 70)
print("SCGPT REPOSITORY")
print("=" * 70)

if not SCGPT_REPO.exists():
    raise FileNotFoundError(
        f"scGPT repository does not exist:\n{SCGPT_REPO}"
    )

# The Python package should be inside the repository.
scgpt_package_dir = SCGPT_REPO / "scgpt"

if not scgpt_package_dir.exists():
    raise FileNotFoundError(
        "The repository exists, but its scgpt package directory "
        f"was not found:\n{scgpt_package_dir}"
    )

# Put the local repository first in Python's import search path.
scgpt_repo_string = str(SCGPT_REPO)

if scgpt_repo_string not in sys.path:
    sys.path.insert(0, scgpt_repo_string)

scgpt_spec = importlib.util.find_spec("scgpt")

print(f"Repository:       {SCGPT_REPO}")
print(f"Package folder:   {scgpt_package_dir}")
print(f"Package detected: {scgpt_spec is not None}")

if scgpt_spec is None:
    raise ImportError(
        "Python could not discover the scgpt package after adding "
        "the repository to sys.path."
    )

print(f"Package origin:   {scgpt_spec.origin}")


# ------------------------------------------------------------
# 4. Inspect VM resources
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VM RESOURCES")
print("=" * 70)

print(f"Python:           {sys.version.split()[0]}")
print(f"Operating system: {platform.platform()}")
print(f"Logical CPUs:     {os.cpu_count()}")

try:
    import psutil

    memory = psutil.virtual_memory()

    print(
        f"Total RAM:        "
        f"{memory.total / 1024**3:.2f} GB"
    )
    print(
        f"Available RAM:    "
        f"{memory.available / 1024**3:.2f} GB"
    )
except ImportError:
    print("RAM:              psutil is not installed")

disk = shutil.disk_usage(PROJECT_DIR)

print(
    f"Disk total:       "
    f"{disk.total / 1024**3:.2f} GB"
)
print(
    f"Disk available:   "
    f"{disk.free / 1024**3:.2f} GB"
)

# Check PyTorch without requiring CUDA.
try:
    import torch

    print(f"PyTorch:          {torch.__version__}")
    print(f"CUDA available:   {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU:              {torch.cuda.get_device_name(0)}")
    else:
        print("Compute plan:     CPU preprocessing on this VM")
except ImportError:
    print("PyTorch:          not importable in this kernel")


# ------------------------------------------------------------
# 5. Check that the remote GEO count file is reachable
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GEO DATA SOURCE")
print("=" * 70)

print(f"Accession:        {GEO_ACCESSION}")
print(f"GEO record:       {GEO_SERIES_URL}")
print(f"Counts URL:       {RAW_COUNTS_URL}")
print(f"Local target:     {RAW_COUNTS_FILE}")


def inspect_remote_file(url: str) -> dict:
    """
    Request only the beginning of a remote file.

    This verifies accessibility without intentionally downloading
    the entire file.
    """
    request = Request(
        url,
        headers={
            "Range": "bytes=0-0",
            "User-Agent": "Mozilla/5.0 GSE200164-research-pipeline",
        },
    )

    try:
        with urlopen(request, timeout=30) as response:
            headers = dict(response.headers.items())

            return {
                "reachable": True,
                "status": getattr(response, "status", None),
                "content_length": headers.get("Content-Length"),
                "content_range": headers.get("Content-Range"),
                "last_modified": headers.get("Last-Modified"),
                "content_type": headers.get("Content-Type"),
            }

    except HTTPError as error:
        # A server may reject range requests while still hosting the file.
        return {
            "reachable": False,
            "error_type": "HTTPError",
            "status": error.code,
            "message": str(error),
        }

    except URLError as error:
        return {
            "reachable": False,
            "error_type": "URLError",
            "message": str(error.reason),
        }


remote_info = inspect_remote_file(RAW_COUNTS_URL)

for key, value in remote_info.items():
    print(f"{key:18s}: {value}")


# ------------------------------------------------------------
# 6. Create a reproducible project configuration
# ------------------------------------------------------------

CONFIG = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "geo_accession": GEO_ACCESSION,
    "geo_series_url": GEO_SERIES_URL,
    "raw_counts_url": RAW_COUNTS_URL,
    "lognorm_url": LOGNORM_URL,
    "project_dir": str(PROJECT_DIR),
    "scgpt_repo": str(SCGPT_REPO),
    "directories": {
        name: str(path)
        for name, path in DIRS.items()
    },
    "raw_counts_file": str(RAW_COUNTS_FILE),
    "remote_file_check": remote_info,
    "processing_device": "cpu",
    "embedding_device_planned": "gpu_colab",
}

CONFIG_FILE = PROJECT_DIR / "project_config.json"

with open(CONFIG_FILE, "w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

checks = {
    "project_directory_ready": PROJECT_DIR.exists(),
    "scgpt_repository_found": SCGPT_REPO.exists(),
    "scgpt_package_found": scgpt_package_dir.exists(),
    "scgpt_python_discoverable": scgpt_spec is not None,
    "geo_counts_url_reachable": remote_info["reachable"],
}

for name, passed in checks.items():
    symbol = "PASS" if passed else "FAIL"
    print(f"{symbol:4s}  {name}")

print(f"\nConfiguration saved to:\n{CONFIG_FILE}")

if all(checks.values()):
    print("\nREADY_FOR_CELL_2 = True")
else:
    print("\nREADY_FOR_CELL_2 = False")
    print("Review any failed checks before downloading the matrix.")

PROJECT PATHS
project     : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory
raw         : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/raw
interim     : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/interim
processed   : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/processed
metadata    : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/metadata
models      : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/models
embeddings  : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/embeddings
figures     : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/figures
tables      : /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/tables
logs        : /home/amithsiri/projects/Summer-Project-20

In [2]:
# ============================================================
# CELL 2: Download and validate the GSE200164 raw-count matrix
# ============================================================
#
# Purpose
# -------
# This cell:
#   1. Defines the raw-count file location.
#   2. Checks that enough disk space is available.
#   3. Downloads the file with resume support.
#   4. Verifies that the gzip archive is not corrupted.
#   5. Computes a SHA-256 checksum for reproducibility.
#   6. Reads only the first few compressed lines to inspect the
#      CSV structure without loading the entire matrix into RAM.
#
# Important
# ---------
# Do NOT use pandas.read_csv() on the complete file yet.
#
# The compressed file is about 1 GB, but the decompressed dense
# CSV is much larger. Loading it directly with pandas could use
# far more memory than the final sparse matrix.
# ============================================================


# ------------------------------------------------------------
# 1. Import the standard-library modules needed in this cell
# ------------------------------------------------------------

from pathlib import Path
import csv
import gzip
import hashlib
import json
import shutil
import subprocess
import time


# ------------------------------------------------------------
# 2. Re-establish the important paths
# ------------------------------------------------------------
#
# These variables may already exist from Cell 1.
#
# We define them again so this cell remains usable even if:
#   - the notebook kernel was restarted;
#   - Cell 1 was not executed in the current session;
#   - the notebook was reopened later.
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
LOG_DIR = PROJECT_DIR / "logs"

# Create the directories if they do not already exist.
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Define the official GEO supplementary-file URL
# ------------------------------------------------------------

RAW_COUNTS_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE200nnn/GSE200164/suppl/"
    "GSE200164_counts.csv.gz"
)

RAW_COUNTS_FILE = RAW_DATA_DIR / "GSE200164_counts.csv.gz"

# curl may temporarily use this partial file while downloading.
PARTIAL_FILE = Path(str(RAW_COUNTS_FILE) + ".part")


# ------------------------------------------------------------
# 4. Display the download configuration
# ------------------------------------------------------------

print("=" * 72)
print("GSE200164 RAW-COUNT DOWNLOAD")
print("=" * 72)

print(f"Remote URL:      {RAW_COUNTS_URL}")
print(f"Destination:     {RAW_COUNTS_FILE}")
print(f"Partial file:    {PARTIAL_FILE}")


# ------------------------------------------------------------
# 5. Check available disk space
# ------------------------------------------------------------
#
# The compressed file is approximately 1 GB.
#
# Later processing will also require room for:
#   - the decompressed stream;
#   - sparse arrays;
#   - the AnnData .h5ad file;
#   - temporary conversion files.
#
# We require at least 10 GB free before starting.
# More than 20 GB free would be preferable.
# ------------------------------------------------------------

disk_usage = shutil.disk_usage(PROJECT_DIR)
free_disk_gb = disk_usage.free / (1024**3)

print(f"Available disk:  {free_disk_gb:.2f} GB")

MINIMUM_FREE_DISK_GB = 10

if free_disk_gb < MINIMUM_FREE_DISK_GB:
    raise RuntimeError(
        f"Only {free_disk_gb:.2f} GB of free disk space is available. "
        f"At least {MINIMUM_FREE_DISK_GB} GB is required before "
        "downloading and converting the matrix."
    )


# ------------------------------------------------------------
# 6. Check whether curl is installed
# ------------------------------------------------------------
#
# curl is preferable here because:
#   - it supports interrupted-download resumption;
#   - it displays download progress;
#   - it handles large files efficiently;
#   - it does not load the file into Python memory.
# ------------------------------------------------------------

curl_path = shutil.which("curl")

if curl_path is None:
    raise RuntimeError(
        "curl was not found on this VM. Install it with:\n\n"
        "sudo apt-get update && sudo apt-get install -y curl"
    )

print(f"curl location:   {curl_path}")


# ------------------------------------------------------------
# 7. Download the file with resume support
# ------------------------------------------------------------
#
# curl options:
#
#   --location
#       Follow redirects.
#
#   --fail
#       Return an error for failed HTTP requests.
#
#   --retry 5
#       Retry temporary network failures up to five times.
#
#   --retry-delay 5
#       Wait five seconds between retry attempts.
#
#   --continue-at -
#       Resume from the current partial-file size.
#
#   --output
#       Write directly to the .part file.
#
# We download to a temporary ".part" file first.
# Only after the download succeeds do we rename it to the final
# filename. This prevents an incomplete file from appearing valid.
# ------------------------------------------------------------

if RAW_COUNTS_FILE.exists():
    existing_size_gb = RAW_COUNTS_FILE.stat().st_size / (1024**3)

    print(
        "\nA completed-looking file already exists:"
        f"\n  {RAW_COUNTS_FILE}"
        f"\n  Size: {existing_size_gb:.3f} GB"
    )

    print(
        "\nThe download step will be skipped. "
        "The existing file will still be validated."
    )

else:
    print("\nStarting or resuming download...\n")

    download_command = [
        curl_path,
        "--location",
        "--fail",
        "--retry", "5",
        "--retry-delay", "5",
        "--continue-at", "-",
        "--output", str(PARTIAL_FILE),
        RAW_COUNTS_URL,
    ]

    download_start_time = time.time()

    # subprocess.run streams curl's progress output directly into
    # the notebook rather than storing it in Python memory.
    completed_process = subprocess.run(
        download_command,
        check=False,
    )

    if completed_process.returncode != 0:
        raise RuntimeError(
            "The download did not complete successfully.\n"
            f"curl exit code: {completed_process.returncode}\n\n"
            "The partial file has been retained, so rerunning this "
            "cell should resume the download."
        )

    download_minutes = (time.time() - download_start_time) / 60

    # Rename the successfully downloaded partial file.
    PARTIAL_FILE.replace(RAW_COUNTS_FILE)

    print(
        f"\nDownload completed in {download_minutes:.2f} minutes."
    )


# ------------------------------------------------------------
# 8. Confirm that the final file exists and is nonempty
# ------------------------------------------------------------

if not RAW_COUNTS_FILE.exists():
    raise FileNotFoundError(
        f"The expected file was not created:\n{RAW_COUNTS_FILE}"
    )

compressed_size_bytes = RAW_COUNTS_FILE.stat().st_size
compressed_size_gb = compressed_size_bytes / (1024**3)

if compressed_size_bytes == 0:
    raise RuntimeError("The downloaded file has a size of zero bytes.")

print("\n" + "=" * 72)
print("DOWNLOADED FILE")
print("=" * 72)

print(f"Path:             {RAW_COUNTS_FILE}")
print(f"Compressed size:  {compressed_size_gb:.3f} GB")


# ------------------------------------------------------------
# 9. Validate the complete gzip archive
# ------------------------------------------------------------
#
# `gzip -t` reads through the compressed file and verifies its CRC
# checksum and compressed structure.
#
# It does not create a decompressed copy on disk.
#
# This may take a little time because it must scan the full archive.
# ------------------------------------------------------------

gzip_executable = shutil.which("gzip")

if gzip_executable is None:
    raise RuntimeError(
        "The system gzip utility was not found."
    )

print("\nValidating the gzip archive...")

gzip_test = subprocess.run(
    [gzip_executable, "-t", str(RAW_COUNTS_FILE)],
    capture_output=True,
    text=True,
)

if gzip_test.returncode != 0:
    raise RuntimeError(
        "The gzip integrity test failed.\n\n"
        f"stderr:\n{gzip_test.stderr}\n\n"
        "Delete the damaged file and rerun this cell."
    )

print("Gzip integrity:   PASS")


# ------------------------------------------------------------
# 10. Compute a SHA-256 checksum
# ------------------------------------------------------------
#
# A checksum provides a reproducible identifier for this exact file.
#
# If the file is transferred to Colab or another machine later,
# computing the checksum again lets us confirm that the transfer did
# not alter the file.
#
# The file is read in 8 MB chunks, so memory usage remains small.
# ------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate the SHA-256 checksum of a file incrementally.

    Parameters
    ----------
    file_path
        File for which the checksum should be calculated.

    chunk_size
        Number of bytes read into memory at one time.

    Returns
    -------
    str
        Hexadecimal SHA-256 digest.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


print("\nCalculating SHA-256 checksum...")

sha256_digest = calculate_sha256(RAW_COUNTS_FILE)

print(f"SHA-256:          {sha256_digest}")


# ------------------------------------------------------------
# 11. Save download metadata
# ------------------------------------------------------------
#
# This JSON record documents:
#   - the source URL;
#   - the local file;
#   - its size;
#   - its checksum;
#   - whether gzip validation passed.
# ------------------------------------------------------------

download_metadata = {
    "geo_accession": "GSE200164",
    "source_url": RAW_COUNTS_URL,
    "local_file": str(RAW_COUNTS_FILE),
    "compressed_size_bytes": compressed_size_bytes,
    "compressed_size_gb": compressed_size_gb,
    "sha256": sha256_digest,
    "gzip_integrity_passed": True,
}

DOWNLOAD_METADATA_FILE = (
    LOG_DIR / "GSE200164_counts_download_metadata.json"
)

with open(
    DOWNLOAD_METADATA_FILE,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        download_metadata,
        file_handle,
        indent=2,
    )

print(
    "\nDownload metadata saved to:"
    f"\n{DOWNLOAD_METADATA_FILE}"
)


# ------------------------------------------------------------
# 12. Inspect only the first few lines of the compressed CSV
# ------------------------------------------------------------
#
# gzip.open() decompresses the file as a stream.
#
# We stop after five rows, so the complete matrix is never loaded.
#
# This inspection will tell us:
#   - whether genes or cells are stored as rows;
#   - what the first column represents;
#   - how cell barcodes are encoded;
#   - approximately how many columns are present.
# ------------------------------------------------------------

NUMBER_OF_ROWS_TO_PREVIEW = 5

print("\n" + "=" * 72)
print("STREAMING CSV PREVIEW")
print("=" * 72)

preview_rows = []

with gzip.open(
    RAW_COUNTS_FILE,
    mode="rt",
    encoding="utf-8",
    newline="",
) as compressed_file:

    csv_reader = csv.reader(compressed_file)

    for row_number, row in enumerate(csv_reader):
        preview_rows.append(row)

        # Print structural information rather than the full row.
        # A complete row may contain tens of thousands of values.
        print(
            f"Row {row_number}: "
            f"{len(row):,} columns; "
            f"first 8 values = {row[:8]}"
        )

        if row_number + 1 >= NUMBER_OF_ROWS_TO_PREVIEW:
            break


# ------------------------------------------------------------
# 13. Summarize the header structure
# ------------------------------------------------------------

if len(preview_rows) == 0:
    raise RuntimeError(
        "The gzip file opened successfully but no CSV rows were found."
    )

header = preview_rows[0]

print("\n" + "=" * 72)
print("HEADER SUMMARY")
print("=" * 72)

print(f"Header columns:   {len(header):,}")
print(f"First header:     {header[0]!r}")
print(f"Next headers:     {header[1:6]}")

# If the first column contains gene names and the remaining columns
# are cells, then the number of cells is approximately:
estimated_cell_columns = max(len(header) - 1, 0)

print(
    f"Possible cells:   {estimated_cell_columns:,} "
    "(assuming column 1 stores gene identifiers)"
)


# ------------------------------------------------------------
# 14. Final status
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("CELL 2 SUMMARY")
print("=" * 72)

print("PASS  Raw-count file exists")
print("PASS  File is nonempty")
print("PASS  Gzip integrity test")
print("PASS  SHA-256 checksum calculated")
print("PASS  CSV header preview completed")

print("\nREADY_FOR_CELL_3 = True")
print(
    "\nNext step: determine the exact matrix orientation and convert "
    "the compressed CSV to a disk-backed sparse matrix without "
    "constructing the full dense matrix in memory."
)

GSE200164 RAW-COUNT DOWNLOAD
Remote URL:      https://ftp.ncbi.nlm.nih.gov/geo/series/GSE200nnn/GSE200164/suppl/GSE200164_counts.csv.gz
Destination:     /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/raw/GSE200164_counts.csv.gz
Partial file:    /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/raw/GSE200164_counts.csv.gz.part
Available disk:  37.36 GB
curl location:   /usr/bin/curl

Starting or resuming download...



  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1060M  100 1060M    0     0   112M      0  0:00:09  0:00:09 --:--:--  110M



Download completed in 0.16 minutes.

DOWNLOADED FILE
Path:             /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/raw/GSE200164_counts.csv.gz
Compressed size:  1.035 GB

Validating the gzip archive...
Gzip integrity:   PASS

Calculating SHA-256 checksum...
SHA-256:          8dbfde2cd00cde69e41051c5d6f9c40dcf7b32c57bbce6ff9bd37ad1e2c5b04f

Download metadata saved to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/logs/GSE200164_counts_download_metadata.json

STREAMING CSV PREVIEW
Row 0: 70,397 columns; first 8 values = ['', 'chromosome', 'start', 'end', 'width', 'strand', 'A1_AAACCTGGTTGGTGGA-1', 'A1_AAACGGGAGAGACGAA-1']
Row 1: 70,397 columns; first 8 values = ['A1BG', 'chr19', '58345178', '58353492', '8315', '-', '0', '0']
Row 2: 70,397 columns; first 8 values = ['A1BG-AS1', 'chr19', '58347718', '58355455', '7738', '+', '0', '0']
Row 3: 70,397 columns; first 8 values = ['A2M', 'chr12', '9067664', '9116229', '48

In [3]:
# ============================================================
# CORRECTED CELL 3: Convert GEO matrix without truncating values
# ============================================================
#
# Critical correction:
# --------------------
# The GEO "counts" matrix contains fractional values.
#
# Therefore:
#   - values must be stored as float32;
#   - values must not be converted to int32;
#   - zeros remain sparse;
#   - all nonzero fractional values must be preserved.
# ============================================================


# ------------------------------------------------------------
# 1. Import packages
# ------------------------------------------------------------

from pathlib import Path
import gc
import gzip
import csv
import json
import time

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse import save_npz


# ------------------------------------------------------------
# 2. Define paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

RAW_COUNTS_FILE = (
    PROJECT_DIR
    / "data/raw/GSE200164_counts.csv.gz"
)

INTERIM_DIR = PROJECT_DIR / "data/interim"
LOG_DIR = PROJECT_DIR / "logs"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

GENES_BY_CELLS_FILE = (
    INTERIM_DIR
    / "GSE200164_expression_genes_by_cells_csr_float32.npz"
)

CELLS_BY_GENES_FILE = (
    INTERIM_DIR
    / "GSE200164_expression_cells_by_genes_csr_float32.npz"
)

GENES_FILE = (
    INTERIM_DIR
    / "GSE200164_genes.csv"
)

BARCODES_FILE = (
    INTERIM_DIR
    / "GSE200164_barcodes.txt"
)

CONVERSION_LOG_FILE = (
    LOG_DIR
    / "GSE200164_float32_sparse_conversion.json"
)


# ------------------------------------------------------------
# 3. Define the six annotation columns
# ------------------------------------------------------------

N_ANNOTATION_COLUMNS = 6

ANNOTATION_NAMES = [
    "gene_symbol",
    "chromosome",
    "start",
    "end",
    "width",
    "strand",
]


# ------------------------------------------------------------
# 4. Read the header and recover cell barcodes
# ------------------------------------------------------------

with gzip.open(
    RAW_COUNTS_FILE,
    mode="rt",
    encoding="utf-8",
    newline="",
) as file_handle:

    reader = csv.reader(file_handle)
    header = next(reader)

cell_barcodes = header[N_ANNOTATION_COLUMNS:]

n_cells = len(cell_barcodes)

print(f"Cells in source matrix: {n_cells:,}")


# ------------------------------------------------------------
# 5. Count gene rows
# ------------------------------------------------------------

print("Counting gene rows...")

with gzip.open(
    RAW_COUNTS_FILE,
    mode="rt",
    encoding="utf-8",
) as file_handle:

    n_lines = sum(1 for _ in file_handle)

n_genes_expected = n_lines - 1

print(f"Genes in source matrix: {n_genes_expected:,}")


# ------------------------------------------------------------
# 6. Save the barcode order
# ------------------------------------------------------------

with open(
    BARCODES_FILE,
    mode="w",
    encoding="utf-8",
) as file_handle:

    for barcode in cell_barcodes:
        file_handle.write(f"{barcode}\n")


# ------------------------------------------------------------
# 7. Prepare chunked CSV reading
# ------------------------------------------------------------
#
# float32 preserves decimal values while using half the memory
# of float64.
# ------------------------------------------------------------

GENES_PER_CHUNK = 100

column_names = ANNOTATION_NAMES + cell_barcodes

chunk_iterator = pd.read_csv(
    RAW_COUNTS_FILE,
    compression="gzip",
    header=0,
    names=column_names,
    chunksize=GENES_PER_CHUNK,
    engine="c",
    low_memory=False,
)


# ------------------------------------------------------------
# 8. Stream and convert each chunk
# ------------------------------------------------------------

sparse_blocks = []
gene_metadata_blocks = []

genes_processed = 0
total_nonzero = 0
minimum_value = np.inf
maximum_value = -np.inf
fractional_nonzero_values = 0

start_time = time.time()

for chunk_number, chunk in enumerate(
    chunk_iterator,
    start=1,
):

    # Save the six gene annotation columns.
    annotation_chunk = chunk.iloc[
        :,
        :N_ANNOTATION_COLUMNS,
    ].copy()

    annotation_chunk.columns = ANNOTATION_NAMES

    gene_metadata_blocks.append(annotation_chunk)

    # Extract expression values as float32.
    expression_array = chunk.iloc[
        :,
        N_ANNOTATION_COLUMNS:,
    ].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    # Validate that all values are finite.
    if not np.isfinite(expression_array).all():
        raise ValueError(
            f"Nonfinite values found in chunk {chunk_number}."
        )

    # Expression values must not be negative.
    if np.any(expression_array < 0):
        raise ValueError(
            f"Negative values found in chunk {chunk_number}."
        )

    # Record value-range statistics.
    minimum_value = min(
        minimum_value,
        float(expression_array.min()),
    )

    maximum_value = max(
        maximum_value,
        float(expression_array.max()),
    )

    # Count nonzero values that are genuinely fractional.
    nonzero_values = expression_array[
        expression_array != 0
    ]

    fractional_nonzero_values += int(
        np.count_nonzero(
            nonzero_values
            != np.floor(nonzero_values)
        )
    )

    # Convert the block to sparse format.
    sparse_block = sparse.csr_matrix(
        expression_array,
        dtype=np.float32,
    )

    sparse_block.eliminate_zeros()

    sparse_blocks.append(sparse_block)

    genes_processed += sparse_block.shape[0]
    total_nonzero += sparse_block.nnz

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        f"\rChunk {chunk_number:4d} | "
        f"genes {genes_processed:6,}/"
        f"{n_genes_expected:,} | "
        f"nonzeros {total_nonzero:,} | "
        f"{elapsed_minutes:.1f} min",
        end="",
        flush=True,
    )

    del expression_array
    del nonzero_values
    del chunk

    gc.collect()

print("\n\nFinished streaming the source matrix.")


# ------------------------------------------------------------
# 9. Validate the number of genes processed
# ------------------------------------------------------------

if genes_processed != n_genes_expected:
    raise RuntimeError(
        f"Expected {n_genes_expected:,} genes but processed "
        f"{genes_processed:,}."
    )


# ------------------------------------------------------------
# 10. Combine sparse blocks
# ------------------------------------------------------------

print("Combining sparse blocks...")

genes_by_cells = sparse.vstack(
    sparse_blocks,
    format="csr",
    dtype=np.float32,
)

del sparse_blocks
gc.collect()


# ------------------------------------------------------------
# 11. Combine and save gene metadata
# ------------------------------------------------------------

gene_metadata = pd.concat(
    gene_metadata_blocks,
    axis=0,
    ignore_index=True,
)

del gene_metadata_blocks
gc.collect()

gene_metadata.to_csv(
    GENES_FILE,
    index=False,
)


# ------------------------------------------------------------
# 12. Validate the genes × cells matrix
# ------------------------------------------------------------

expected_shape = (
    n_genes_expected,
    n_cells,
)

if genes_by_cells.shape != expected_shape:
    raise RuntimeError(
        f"Expected shape {expected_shape}, "
        f"observed {genes_by_cells.shape}."
    )

print(f"Genes × cells shape: {genes_by_cells.shape}")
print(f"Nonzero entries:     {genes_by_cells.nnz:,}")
print(f"Data type:           {genes_by_cells.dtype}")


# ------------------------------------------------------------
# 13. Save genes × cells orientation
# ------------------------------------------------------------

print("Saving genes × cells sparse matrix...")

save_npz(
    GENES_BY_CELLS_FILE,
    genes_by_cells,
    compressed=True,
)


# ------------------------------------------------------------
# 14. Transpose to cells × genes
# ------------------------------------------------------------

print("Transposing to cells × genes...")

cells_by_genes = (
    genes_by_cells
    .transpose()
    .tocsr()
    .astype(np.float32)
)

cells_by_genes.sort_indices()
cells_by_genes.eliminate_zeros()

if cells_by_genes.shape != (
    n_cells,
    n_genes_expected,
):
    raise RuntimeError(
        "Unexpected cells × genes matrix shape."
    )

if cells_by_genes.nnz != genes_by_cells.nnz:
    raise RuntimeError(
        "Nonzero count changed during transpose."
    )


# ------------------------------------------------------------
# 15. Save cells × genes orientation
# ------------------------------------------------------------

print("Saving cells × genes sparse matrix...")

save_npz(
    CELLS_BY_GENES_FILE,
    cells_by_genes,
    compressed=True,
)


# ------------------------------------------------------------
# 16. Calculate summary statistics
# ------------------------------------------------------------

values_per_cell = np.asarray(
    cells_by_genes.sum(axis=1)
).ravel()

features_per_cell = np.diff(
    cells_by_genes.indptr
)

matrix_density = (
    cells_by_genes.nnz
    / (
        cells_by_genes.shape[0]
        * cells_by_genes.shape[1]
    )
)


# ------------------------------------------------------------
# 17. Print and save the corrected summary
# ------------------------------------------------------------

conversion_summary = {
    "matrix_shape_cells_by_genes": [
        int(cells_by_genes.shape[0]),
        int(cells_by_genes.shape[1]),
    ],
    "dtype": str(cells_by_genes.dtype),
    "nonzero_entries": int(cells_by_genes.nnz),
    "matrix_density": float(matrix_density),
    "minimum_value": float(minimum_value),
    "maximum_value": float(maximum_value),
    "fractional_nonzero_values": int(
        fractional_nonzero_values
    ),
    "mean_expression_sum_per_cell": float(
        values_per_cell.mean()
    ),
    "median_expression_sum_per_cell": float(
        np.median(values_per_cell)
    ),
    "mean_detected_genes_per_cell": float(
        features_per_cell.mean()
    ),
    "median_detected_genes_per_cell": float(
        np.median(features_per_cell)
    ),
    "genes_by_cells_file": str(
        GENES_BY_CELLS_FILE
    ),
    "cells_by_genes_file": str(
        CELLS_BY_GENES_FILE
    ),
}

with open(
    CONVERSION_LOG_FILE,
    mode="w",
    encoding="utf-8",
) as file_handle:

    json.dump(
        conversion_summary,
        file_handle,
        indent=2,
    )

print("\n" + "=" * 72)
print("CORRECTED FLOAT32 CONVERSION SUMMARY")
print("=" * 72)

for key, value in conversion_summary.items():
    print(f"{key}: {value}")

print("\nREADY_FOR_STRICT_AUDIT = True")

Cells in source matrix: 70,391
Counting gene rows...
Genes in source matrix: 19,689
Chunk  197 | genes 19,689/19,689 | nonzeros 126,603,213 | 9.0 min

Finished streaming the source matrix.
Combining sparse blocks...
Genes × cells shape: (19689, 70391)
Nonzero entries:     126,603,213
Data type:           float32
Saving genes × cells sparse matrix...
Transposing to cells × genes...
Saving cells × genes sparse matrix...

CORRECTED FLOAT32 CONVERSION SUMMARY
matrix_shape_cells_by_genes: [70391, 19689]
dtype: float32
nonzero_entries: 126603213
matrix_density: 0.09134902863044658
minimum_value: 0.0
maximum_value: 2742.825927734375
fractional_nonzero_values: 124456824
mean_expression_sum_per_cell: 5159.91162109375
median_expression_sum_per_cell: 4376.1259765625
mean_detected_genes_per_cell: 1798.5710247048628
median_detected_genes_per_cell: 1691.0
genes_by_cells_file: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/interim/GSE200164_expression_genes_by_ce

In [5]:
# ============================================================
# CELL 3A: Strict spot-check of CSV-to-sparse conversion
# ============================================================
#
# This cell verifies that selected values in the float32 sparse
# matrix match the original compressed GEO CSV after both are
# represented as float32.
#
# It checks:
#   1. Complete cell-barcode ordering.
#   2. Selected gene ordering.
#   3. Random and boundary gene-cell values.
#   4. Full expression vectors for selected genes.
#   5. Per-gene nonzero counts and total expression.
#
# This cell does not modify the matrix or AnnData object.
# ============================================================


# ------------------------------------------------------------
# 1. Import packages
# ------------------------------------------------------------

from pathlib import Path
import csv
import gzip
import random

import numpy as np
import pandas as pd

from scipy.sparse import load_npz, issparse


# ------------------------------------------------------------
# 2. Define paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

RAW_EXPRESSION_FILE = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "GSE200164_counts.csv.gz"
)

SPARSE_EXPRESSION_FILE = (
    PROJECT_DIR
    / "data"
    / "interim"
    / "GSE200164_expression_cells_by_genes_csr_float32.npz"
)

GENES_FILE = (
    PROJECT_DIR
    / "data"
    / "interim"
    / "GSE200164_genes.csv"
)

BARCODES_FILE = (
    PROJECT_DIR
    / "data"
    / "interim"
    / "GSE200164_barcodes.txt"
)

AUDIT_OUTPUT_FILE = (
    PROJECT_DIR
    / "tables"
    / "GSE200164_sparse_conversion_audit.csv"
)

FULL_VECTOR_OUTPUT_FILE = (
    PROJECT_DIR
    / "tables"
    / "GSE200164_sparse_full_vector_audit.csv"
)

AUDIT_OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Set floating-point comparison tolerances
# ------------------------------------------------------------
#
# The sparse matrix was intentionally stored as float32.
#
# We therefore parse the original values as float32 and compare
# them using a small numerical tolerance.
# ------------------------------------------------------------

RELATIVE_TOLERANCE = 1e-6
ABSOLUTE_TOLERANCE = 1e-7


# ------------------------------------------------------------
# 4. Confirm that all required files exist
# ------------------------------------------------------------

required_files = [
    RAW_EXPRESSION_FILE,
    SPARSE_EXPRESSION_FILE,
    GENES_FILE,
    BARCODES_FILE,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{file_path}"
        )

print("All audit input files were found.")


# ------------------------------------------------------------
# 5. Load the sparse matrix and ordered metadata
# ------------------------------------------------------------

print("\nLoading sparse matrix...")

expression = load_npz(
    SPARSE_EXPRESSION_FILE
)

if not issparse(expression):
    raise TypeError(
        "The expression matrix was unexpectedly loaded as dense."
    )

expression = expression.tocsr()

print(f"Sparse shape:      {expression.shape}")
print(f"Sparse nonzeros:   {expression.nnz:,}")
print(f"Sparse dtype:      {expression.dtype}")

if expression.dtype != np.float32:
    raise TypeError(
        "The corrected sparse matrix should have dtype float32, "
        f"but observed {expression.dtype}."
    )


# Load genes in exact matrix-column order.
gene_metadata = pd.read_csv(
    GENES_FILE
)

gene_symbols = (
    gene_metadata["gene_symbol"]
    .astype(str)
    .tolist()
)


# Load cells in exact matrix-row order.
with open(
    BARCODES_FILE,
    mode="r",
    encoding="utf-8",
) as barcode_handle:

    cell_barcodes = [
        line.strip()
        for line in barcode_handle
        if line.strip()
    ]


# Validate matrix dimensions.
if expression.shape[0] != len(cell_barcodes):
    raise ValueError(
        "Sparse matrix row count does not match barcode count."
    )

if expression.shape[1] != len(gene_symbols):
    raise ValueError(
        "Sparse matrix column count does not match gene count."
    )

print(f"Ordered cells:     {len(cell_barcodes):,}")
print(f"Ordered genes:     {len(gene_symbols):,}")


# ------------------------------------------------------------
# 6. Read and validate the original CSV header
# ------------------------------------------------------------
#
# The first six columns contain gene annotations.
# Cell-expression columns begin at index 6.
# ------------------------------------------------------------

N_ANNOTATION_COLUMNS = 6

with gzip.open(
    RAW_EXPRESSION_FILE,
    mode="rt",
    encoding="utf-8",
    newline="",
) as compressed_file:

    csv_reader = csv.reader(
        compressed_file
    )

    original_header = next(
        csv_reader
    )


original_barcodes = original_header[
    N_ANNOTATION_COLUMNS:
]


# Check both length and ordering.
if len(original_barcodes) != len(cell_barcodes):
    raise ValueError(
        "Original CSV barcode count does not match saved barcode count.\n"
        f"CSV barcodes:   {len(original_barcodes):,}\n"
        f"Saved barcodes: {len(cell_barcodes):,}"
    )

if original_barcodes != cell_barcodes:

    mismatch_index = next(
        index
        for index, (original, saved)
        in enumerate(
            zip(
                original_barcodes,
                cell_barcodes,
            )
        )
        if original != saved
    )

    raise ValueError(
        "Cell-barcode ordering does not match the original CSV.\n"
        f"First mismatch index: {mismatch_index}\n"
        f"Original barcode:     {original_barcodes[mismatch_index]}\n"
        f"Saved barcode:        {cell_barcodes[mismatch_index]}"
    )

print(
    "\nPASS: Complete barcode ordering matches the original CSV."
)


# ------------------------------------------------------------
# 7. Select reproducible random and boundary indices
# ------------------------------------------------------------

RANDOM_SEED = 42
N_RANDOM_GENES = 12
N_RANDOM_CELLS = 20

random_generator = random.Random(
    RANDOM_SEED
)


# Include the first and last indices to test matrix boundaries.
selected_gene_indices = {
    0,
    len(gene_symbols) - 1,
}

selected_cell_indices = {
    0,
    len(cell_barcodes) - 1,
}


# Add reproducible random indices.
selected_gene_indices.update(
    random_generator.sample(
        range(len(gene_symbols)),
        N_RANDOM_GENES,
    )
)

selected_cell_indices.update(
    random_generator.sample(
        range(len(cell_barcodes)),
        N_RANDOM_CELLS,
    )
)


selected_gene_indices = sorted(
    selected_gene_indices
)

selected_cell_indices = sorted(
    selected_cell_indices
)

print(
    f"\nSelected genes:    "
    f"{len(selected_gene_indices)}"
)

print(
    f"Selected cells:    "
    f"{len(selected_cell_indices)}"
)

print(
    f"Pair comparisons:  "
    f"{len(selected_gene_indices) * len(selected_cell_indices):,}"
)


# ------------------------------------------------------------
# 8. Recover selected gene rows from the original CSV
# ------------------------------------------------------------
#
# The file is streamed only until the final selected gene is
# reached. The entire CSV is never loaded into memory.
# ------------------------------------------------------------

selected_gene_index_set = set(
    selected_gene_indices
)

maximum_selected_gene_index = max(
    selected_gene_indices
)

original_selected_rows = {}

print(
    "\nReading selected gene rows from the original compressed CSV..."
)

with gzip.open(
    RAW_EXPRESSION_FILE,
    mode="rt",
    encoding="utf-8",
    newline="",
) as compressed_file:

    csv_reader = csv.reader(
        compressed_file
    )

    # Skip the CSV header.
    next(csv_reader)

    for gene_index, row in enumerate(
        csv_reader
    ):

        if gene_index in selected_gene_index_set:
            original_selected_rows[gene_index] = row

        if gene_index >= maximum_selected_gene_index:
            break


if len(original_selected_rows) != len(selected_gene_indices):

    missing_indices = sorted(
        set(selected_gene_indices)
        - set(original_selected_rows)
    )

    raise RuntimeError(
        "Not all selected gene rows were found in the CSV.\n"
        f"Missing indices: {missing_indices}"
    )

print(
    f"Recovered selected rows: "
    f"{len(original_selected_rows)}"
)


# ------------------------------------------------------------
# 9. Verify selected gene ordering
# ------------------------------------------------------------

for gene_index in selected_gene_indices:

    csv_gene_symbol = (
        original_selected_rows[gene_index][0]
    )

    saved_gene_symbol = (
        gene_symbols[gene_index]
    )

    if csv_gene_symbol != saved_gene_symbol:

        raise ValueError(
            "Gene ordering mismatch detected.\n"
            f"Gene index:        {gene_index}\n"
            f"Original CSV gene: {csv_gene_symbol}\n"
            f"Saved matrix gene: {saved_gene_symbol}"
        )

print(
    "PASS: Selected gene ordering matches the original CSV."
)


# ------------------------------------------------------------
# 10. Compare selected gene-cell values
# ------------------------------------------------------------

audit_records = []

number_of_matches = 0
number_of_mismatches = 0


for gene_index in selected_gene_indices:

    original_row = (
        original_selected_rows[gene_index]
    )

    gene_symbol = (
        gene_symbols[gene_index]
    )

    for cell_index in selected_cell_indices:

        cell_barcode = (
            cell_barcodes[cell_index]
        )

        csv_column_index = (
            N_ANNOTATION_COLUMNS
            + cell_index
        )

        # Parse the source value using the same precision used in
        # the corrected sparse matrix.
        original_value = np.float32(
            original_row[csv_column_index]
        )

        sparse_value = np.float32(
            expression[cell_index, gene_index]
        )

        absolute_difference = float(
            abs(
                np.float64(original_value)
                - np.float64(sparse_value)
            )
        )

        values_match = bool(
            np.isclose(
                original_value,
                sparse_value,
                rtol=RELATIVE_TOLERANCE,
                atol=ABSOLUTE_TOLERANCE,
            )
        )

        if values_match:
            number_of_matches += 1
        else:
            number_of_mismatches += 1

        audit_records.append(
            {
                "gene_index": gene_index,
                "gene_symbol": gene_symbol,
                "cell_index": cell_index,
                "cell_barcode": cell_barcode,
                "original_csv_float32": float(original_value),
                "sparse_matrix_float32": float(sparse_value),
                "absolute_difference": absolute_difference,
                "match": values_match,
            }
        )


audit_table = pd.DataFrame(
    audit_records
)

audit_table.to_csv(
    AUDIT_OUTPUT_FILE,
    index=False,
)

print(
    f"\nPairwise audit table saved to:\n"
    f"{AUDIT_OUTPUT_FILE}"
)


# ------------------------------------------------------------
# 11. Report pairwise audit results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PAIRWISE EXPRESSION AUDIT")
print("=" * 72)

print(f"Matches:           {number_of_matches:,}")
print(f"Mismatches:        {number_of_mismatches:,}")

maximum_pair_difference = (
    audit_table["absolute_difference"].max()
)

print(
    f"Maximum difference: "
    f"{maximum_pair_difference:.10g}"
)

if number_of_mismatches > 0:

    mismatch_table = audit_table.loc[
        ~audit_table["match"]
    ]

    display(
        mismatch_table.head(20)
    )

    raise AssertionError(
        "Sparse-conversion pairwise audit failed. "
        f"{number_of_mismatches} selected values did not match."
    )

print(
    "PASS: Every selected CSV value matches the sparse matrix."
)


# ------------------------------------------------------------
# 12. Compare full expression vectors for selected genes
# ------------------------------------------------------------
#
# Each selected gene is checked across all 70,391 cells.
#
# The vectors remain float32. No integer conversion is performed.
# ------------------------------------------------------------

print(
    "\nComparing full expression vectors for selected genes..."
)

full_vector_results = []


for gene_index in selected_gene_indices:

    original_row = (
        original_selected_rows[gene_index]
    )

    # Parse the complete source gene vector as float32.
    original_full_vector = np.asarray(
        original_row[N_ANNOTATION_COLUMNS:],
        dtype=np.float32,
    )

    # Extract the corresponding complete sparse matrix column.
    sparse_full_vector = (
        expression[:, gene_index]
        .toarray()
        .ravel()
        .astype(
            np.float32,
            copy=False,
        )
    )


    # Confirm vector lengths before numerical comparison.
    if original_full_vector.shape != sparse_full_vector.shape:
        raise ValueError(
            "Full-vector lengths do not match.\n"
            f"Gene: {gene_symbols[gene_index]}\n"
            f"CSV shape:    {original_full_vector.shape}\n"
            f"Sparse shape: {sparse_full_vector.shape}"
        )


    difference_vector = np.abs(
        original_full_vector.astype(np.float64)
        - sparse_full_vector.astype(np.float64)
    )

    full_vector_match = bool(
        np.allclose(
            original_full_vector,
            sparse_full_vector,
            rtol=RELATIVE_TOLERANCE,
            atol=ABSOLUTE_TOLERANCE,
        )
    )

    maximum_absolute_difference = float(
        difference_vector.max()
    )

    mismatched_positions = int(
        np.count_nonzero(
            ~np.isclose(
                original_full_vector,
                sparse_full_vector,
                rtol=RELATIVE_TOLERANCE,
                atol=ABSOLUTE_TOLERANCE,
            )
        )
    )


    # Sum using float64 to reduce accumulated rounding error.
    original_total_expression = float(
        original_full_vector.sum(
            dtype=np.float64
        )
    )

    sparse_total_expression = float(
        sparse_full_vector.sum(
            dtype=np.float64
        )
    )

    total_expression_matches = bool(
        np.isclose(
            original_total_expression,
            sparse_total_expression,
            rtol=RELATIVE_TOLERANCE,
            atol=ABSOLUTE_TOLERANCE,
        )
    )


    full_vector_results.append(
        {
            "gene_index": gene_index,
            "gene_symbol": gene_symbols[gene_index],
            "all_cells_match": full_vector_match,
            "mismatched_positions": mismatched_positions,
            "maximum_absolute_difference": (
                maximum_absolute_difference
            ),
            "original_nonzero_cells": int(
                np.count_nonzero(
                    original_full_vector
                )
            ),
            "sparse_nonzero_cells": int(
                np.count_nonzero(
                    sparse_full_vector
                )
            ),
            "original_total_expression": (
                original_total_expression
            ),
            "sparse_total_expression": (
                sparse_total_expression
            ),
            "total_expression_matches": (
                total_expression_matches
            ),
        }
    )


full_vector_table = pd.DataFrame(
    full_vector_results
)

full_vector_table.to_csv(
    FULL_VECTOR_OUTPUT_FILE,
    index=False,
)

display(
    full_vector_table
)

print(
    f"\nFull-vector audit table saved to:\n"
    f"{FULL_VECTOR_OUTPUT_FILE}"
)


# ------------------------------------------------------------
# 13. Validate full-vector results
# ------------------------------------------------------------

if not full_vector_table["all_cells_match"].all():

    failed_vectors = full_vector_table.loc[
        ~full_vector_table["all_cells_match"]
    ]

    raise AssertionError(
        "Full-vector audit failed for one or more genes.\n"
        f"{failed_vectors}"
    )


if not (
    full_vector_table["original_nonzero_cells"]
    == full_vector_table["sparse_nonzero_cells"]
).all():

    raise AssertionError(
        "The number of nonzero cells differs for one or more genes."
    )


if not full_vector_table[
    "total_expression_matches"
].all():

    raise AssertionError(
        "Total expression differs for one or more selected genes."
    )


print(
    "PASS: Every selected gene vector matches across all cells."
)

print(
    "PASS: Selected-gene nonzero counts and total expression match."
)


# ------------------------------------------------------------
# 14. Final audit summary
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("STRICT CONVERSION AUDIT SUMMARY")
print("=" * 72)

print(
    f"Barcode order checked:       "
    f"{len(cell_barcodes):,} cells"
)

print(
    f"Gene rows checked:           "
    f"{len(selected_gene_indices)}"
)

print(
    f"Random/boundary pairs:       "
    f"{len(audit_table):,}"
)

print(
    f"Full gene vectors checked:   "
    f"{len(full_vector_table)}"
)

print(
    f"Pairwise mismatches:         "
    f"{number_of_mismatches}"
)

print(
    f"Full-vector mismatches:      "
    f"{int((~full_vector_table['all_cells_match']).sum())}"
)

print(
    f"Largest pairwise difference: "
    f"{maximum_pair_difference:.10g}"
)

print(
    "\nSTRICT_CONVERSION_CHECK = PASS"
)

print(
    "\nThe audited float32 sparse values match the original GEO "
    "expression values within the declared numerical tolerance."
)

All audit input files were found.

Loading sparse matrix...
Sparse shape:      (70391, 19689)
Sparse nonzeros:   126,603,213
Sparse dtype:      float32
Ordered cells:     70,391
Ordered genes:     19,689

PASS: Complete barcode ordering matches the original CSV.

Selected genes:    14
Selected cells:    22
Pair comparisons:  308

Reading selected gene rows from the original compressed CSV...
Recovered selected rows: 14
PASS: Selected gene ordering matches the original CSV.

Pairwise audit table saved to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/tables/GSE200164_sparse_conversion_audit.csv

PAIRWISE EXPRESSION AUDIT
Matches:           308
Mismatches:        0
Maximum difference: 0
PASS: Every selected CSV value matches the sparse matrix.

Comparing full expression vectors for selected genes...


,gene_index,gene_symbol,all_cells_match,mismatched_positions,maximum_absolute_difference,original_nonzero_cells,sparse_nonzero_cells,original_total_expression,sparse_total_expression,total_expression_matches
0,0,A1BG,True,0,0.0,14837,14837,18113.753503,18113.753503,True
1,819,AC012065.4,True,0,0.0,28,28,28.000000,28.000000,True
2,1041,AC021087.1,True,0,0.0,102,102,103.628750,103.628750,True
3,2848,AL022322.1,True,0,0.0,18,18,18.000000,18.000000,True
4,3358,AL355075.2,True,0,0.0,3,3,3.000000,3.000000,True
5,3648,AL592494.2,True,0,0.0,132,132,128.510448,128.510448,True
6,4572,ATP9B,True,0,0.0,7821,7821,8238.862120,8238.862120,True
7,7314,ECSCR,True,0,0.0,24,24,24.784744,24.784744,True
8,8024,FBRSL1,True,0,0.0,9343,9343,10404.703865,10404.703865,True
9,9012,HAVCR2,True,0,0.0,6546,6546,9862.744664,9862.744664,True



Full-vector audit table saved to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/tables/GSE200164_sparse_full_vector_audit.csv
PASS: Every selected gene vector matches across all cells.
PASS: Selected-gene nonzero counts and total expression match.

STRICT CONVERSION AUDIT SUMMARY
Barcode order checked:       70,391 cells
Gene rows checked:           14
Random/boundary pairs:       308
Full gene vectors checked:   14
Pairwise mismatches:         0
Full-vector mismatches:      0
Largest pairwise difference: 0

STRICT_CONVERSION_CHECK = PASS

The audited float32 sparse values match the original GEO expression values within the declared numerical tolerance.


In [2]:
# ============================================================
# CORRECTED CELL 4:
# Construct and safely save the initial AnnData object
# ============================================================
#
# Correction:
# -----------
# Do not fill text metadata columns with pd.NA before writing H5AD.
# h5py cannot always serialize mixed object columns containing
# pandas.NA values.
#
# Text placeholders are stored as ordinary strings ("unknown").
# Numeric missing values remain np.nan.
# ============================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
from scipy.sparse import load_npz, issparse
import anndata as ad


# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
LOG_DIR = PROJECT_DIR / "logs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

EXPRESSION_FILE = (
    INTERIM_DIR
    / "GSE200164_expression_cells_by_genes_csr_float32.npz"
)

GENES_FILE = (
    INTERIM_DIR
    / "GSE200164_genes.csv"
)

BARCODES_FILE = (
    INTERIM_DIR
    / "GSE200164_barcodes.txt"
)

OUTPUT_H5AD = (
    PROCESSED_DIR
    / "GSE200164_expression_unannotated.h5ad"
)

SUMMARY_FILE = (
    LOG_DIR
    / "GSE200164_anndata_construction_summary.json"
)


# ------------------------------------------------------------
# 2. Confirm required files exist
# ------------------------------------------------------------

required_files = [
    EXPRESSION_FILE,
    GENES_FILE,
    BARCODES_FILE,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{file_path}"
        )

print("PASS: All AnnData input files were found.")


# ------------------------------------------------------------
# 3. Load sparse cells × genes expression matrix
# ------------------------------------------------------------

print("\nLoading sparse expression matrix...")

expression = load_npz(EXPRESSION_FILE)

if not issparse(expression):
    raise TypeError(
        "Expression matrix unexpectedly loaded as dense."
    )

expression = expression.tocsr()

if expression.dtype != np.float32:
    expression = expression.astype(
        np.float32,
        copy=False,
    )

expression.sort_indices()
expression.eliminate_zeros()

print(f"Expression shape:    {expression.shape}")
print(f"Expression dtype:    {expression.dtype}")
print(f"Expression nonzeros: {expression.nnz:,}")


# ------------------------------------------------------------
# 4. Load gene metadata
# ------------------------------------------------------------

gene_metadata = pd.read_csv(
    GENES_FILE,
    low_memory=False,
)

required_gene_columns = [
    "gene_symbol",
    "chromosome",
    "start",
    "end",
    "width",
    "strand",
]

missing_gene_columns = [
    column
    for column in required_gene_columns
    if column not in gene_metadata.columns
]

if missing_gene_columns:
    raise ValueError(
        "Missing expected gene metadata columns:\n"
        f"{missing_gene_columns}"
    )


# ------------------------------------------------------------
# 5. Convert gene metadata to safe, consistent dtypes
# ------------------------------------------------------------

# These columns should be stored as ordinary strings.
for column in [
    "gene_symbol",
    "chromosome",
    "strand",
]:
    gene_metadata[column] = (
        gene_metadata[column]
        .fillna("unknown")
        .astype(str)
    )

# These columns are numeric.
for column in [
    "start",
    "end",
    "width",
]:
    gene_metadata[column] = pd.to_numeric(
        gene_metadata[column],
        errors="coerce",
    )

print(f"Gene metadata rows: {len(gene_metadata):,}")


# ------------------------------------------------------------
# 6. Load cell barcodes in matrix-row order
# ------------------------------------------------------------

with open(
    BARCODES_FILE,
    mode="r",
    encoding="utf-8",
) as file_handle:
    cell_barcodes = [
        line.strip()
        for line in file_handle
        if line.strip()
    ]

print(f"Cell barcodes: {len(cell_barcodes):,}")


# ------------------------------------------------------------
# 7. Validate dimensions
# ------------------------------------------------------------

n_cells, n_genes = expression.shape

if n_cells != len(cell_barcodes):
    raise ValueError(
        "Matrix rows do not match cell-barcode count.\n"
        f"Matrix rows: {n_cells:,}\n"
        f"Barcodes:    {len(cell_barcodes):,}"
    )

if n_genes != len(gene_metadata):
    raise ValueError(
        "Matrix columns do not match gene-metadata rows.\n"
        f"Matrix genes: {n_genes:,}\n"
        f"Metadata:     {len(gene_metadata):,}"
    )

print("PASS: Matrix dimensions match barcode and gene metadata.")


# ------------------------------------------------------------
# 8. Validate barcode uniqueness
# ------------------------------------------------------------

barcode_index = pd.Index(
    cell_barcodes,
    dtype="object",
)

if not barcode_index.is_unique:
    duplicated_barcodes = (
        barcode_index[
            barcode_index.duplicated()
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "Duplicate cell barcodes were found.\n"
        f"Examples: {duplicated_barcodes[:10]}"
    )

print("PASS: Cell barcodes are unique.")


# ------------------------------------------------------------
# 9. Construct preliminary cell metadata
# ------------------------------------------------------------

obs = pd.DataFrame(
    index=pd.Index(
        cell_barcodes,
        name="cell_barcode",
        dtype="object",
    )
)

# Barcode format:
#
# A1_AAACCTGGTTGGTGGA-1
#
# Split only at the first underscore.
barcode_series = pd.Series(
    cell_barcodes,
    index=obs.index,
    dtype="object",
)

barcode_parts = barcode_series.str.split(
    "_",
    n=1,
    expand=True,
)

if barcode_parts.shape[1] != 2:
    raise ValueError(
        "Cell barcodes did not split into exactly two components."
    )

obs["sample_prefix"] = (
    barcode_parts.iloc[:, 0]
    .fillna("unknown")
    .astype(str)
    .to_numpy()
)

obs["tenx_barcode"] = (
    barcode_parts.iloc[:, 1]
    .fillna("unknown")
    .astype(str)
    .to_numpy()
)

# Safe placeholders.
#
# Do not use pd.NA here because it caused HDF5 serialization
# to fail.
obs["patient_id"] = np.full(
    n_cells,
    "unknown",
    dtype=object,
)

obs["diagnosis"] = np.full(
    n_cells,
    "unknown",
    dtype=object,
)

obs["sex"] = np.full(
    n_cells,
    "unknown",
    dtype=object,
)

obs["cell_type"] = np.full(
    n_cells,
    "unknown",
    dtype=object,
)

# Numeric missing values can remain NaN.
obs["age"] = np.full(
    n_cells,
    np.nan,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 10. Force all text columns into safe string arrays
# ------------------------------------------------------------

text_obs_columns = [
    "sample_prefix",
    "tenx_barcode",
    "patient_id",
    "diagnosis",
    "sex",
    "cell_type",
]

for column in text_obs_columns:
    obs[column] = (
        obs[column]
        .fillna("unknown")
        .astype(str)
    )

print("\nCell metadata dtypes")
print("--------------------")
print(obs.dtypes)


# ------------------------------------------------------------
# 11. Construct AnnData gene metadata
# ------------------------------------------------------------

var = gene_metadata.copy()

# Preserve the original gene symbol explicitly.
var["gene_symbol"] = (
    var["gene_symbol"]
    .fillna("unknown")
    .astype(str)
)

# Use gene symbols as initial variable names.
var.index = pd.Index(
    var["gene_symbol"].tolist(),
    name="gene_id",
    dtype="object",
)


# ------------------------------------------------------------
# 12. Make variable names unique
# ------------------------------------------------------------

# AnnData's built-in method is safer and simpler than manually
# constructing duplicate suffixes.
temporary_adata = ad.AnnData(
    X=expression,
    obs=obs,
    var=var,
)

temporary_adata.var_names_make_unique(
    join="-duplicate-"
)

adata = temporary_adata
del temporary_adata


# ------------------------------------------------------------
# 13. Add dataset provenance
# ------------------------------------------------------------

adata.uns["dataset"] = {
    "geo_accession": "GSE200164",
    "source_filename": "GSE200164_counts.csv.gz",
    "matrix_description": (
        "Audited GEO expression matrix stored as sparse float32."
    ),
    "expression_status": (
        "Fractional values detected; untreated raw UMI counts "
        "are not assumed."
    ),
}

adata.uns["processing"] = {
    "matrix_orientation": "cells_by_genes",
    "expression_modified": False,
    "normalization_applied_by_this_pipeline": False,
    "log_transform_applied_by_this_pipeline": False,
    "cell_metadata_attached": False,
}


# ------------------------------------------------------------
# 14. Validate the AnnData object
# ------------------------------------------------------------

print("\nAnnData object")
print("--------------")
print(adata)

if adata.n_obs != n_cells:
    raise AssertionError(
        "AnnData cell count changed unexpectedly."
    )

if adata.n_vars != n_genes:
    raise AssertionError(
        "AnnData gene count changed unexpectedly."
    )

if not issparse(adata.X):
    raise AssertionError(
        "AnnData.X unexpectedly became dense."
    )

if adata.X.dtype != np.float32:
    raise AssertionError(
        f"Expected float32 but observed {adata.X.dtype}."
    )

if adata.obs_names.tolist() != cell_barcodes:
    raise AssertionError(
        "AnnData barcode order no longer matches source order."
    )

if not adata.obs_names.is_unique:
    raise AssertionError(
        "AnnData cell names are not unique."
    )

if not adata.var_names.is_unique:
    raise AssertionError(
        "AnnData gene identifiers are not unique."
    )

print("PASS: AnnData structure validated.")


# ------------------------------------------------------------
# 15. Validate that obs contains no problematic missing objects
# ------------------------------------------------------------

print("\nHDF5 metadata safety audit")
print("--------------------------")

for column in adata.obs.columns:
    dtype = adata.obs[column].dtype
    missing_count = int(
        adata.obs[column].isna().sum()
    )

    print(
        f"{column:16s} "
        f"dtype={str(dtype):12s} "
        f"missing={missing_count:,}"
    )

    if column in text_obs_columns:
        bad_values = adata.obs[column].map(
            lambda value: not isinstance(value, str)
        )

        if bad_values.any():
            examples = (
                adata.obs.loc[
                    bad_values,
                    column,
                ]
                .head()
                .tolist()
            )

            raise TypeError(
                f"Column {column!r} contains non-string objects.\n"
                f"Examples: {examples}"
            )

print("PASS: Text metadata columns contain only Python strings.")


# ------------------------------------------------------------
# 16. Report sample-prefix structure
# ------------------------------------------------------------

prefix_counts = (
    adata.obs["sample_prefix"]
    .value_counts()
    .sort_index()
)

print("\nSample-prefix summary")
print("---------------------")
print(f"Number of prefixes: {prefix_counts.shape[0]:,}")

print("\nFirst 20 prefixes:")
print(prefix_counts.head(20))

print("\nCells per prefix:")
print(prefix_counts.describe())


# ------------------------------------------------------------
# 17. Remove any incomplete output from the failed write
# ------------------------------------------------------------

if OUTPUT_H5AD.exists():
    print(
        "\nRemoving output left by the previous failed write:"
        f"\n{OUTPUT_H5AD}"
    )

    OUTPUT_H5AD.unlink()


# ------------------------------------------------------------
# 18. Save AnnData
# ------------------------------------------------------------

print(f"\nSaving AnnData to:\n{OUTPUT_H5AD}")

adata.write_h5ad(
    filename=OUTPUT_H5AD,
    compression="gzip",
    convert_strings_to_categoricals=False,
)

if not OUTPUT_H5AD.exists():
    raise FileNotFoundError(
        "AnnData write completed without creating the output file."
    )

output_size_gb = (
    OUTPUT_H5AD.stat().st_size
    / (1024**3)
)

print(f"Saved file size: {output_size_gb:.3f} GB")


# ------------------------------------------------------------
# 19. Read the H5AD back from disk
# ------------------------------------------------------------

print("\nReading the saved H5AD back for validation...")

adata_check = ad.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

print(adata_check)

if adata_check.shape != adata.shape:
    raise AssertionError(
        "Saved H5AD shape differs from in-memory AnnData shape."
    )

if adata_check.obs_names.tolist() != adata.obs_names.tolist():
    raise AssertionError(
        "Saved H5AD cell order differs from in-memory cell order."
    )

if adata_check.var_names.tolist() != adata.var_names.tolist():
    raise AssertionError(
        "Saved H5AD gene order differs from in-memory gene order."
    )

adata_check.file.close()

print("PASS: Saved H5AD reopened with matching dimensions and order.")


# ------------------------------------------------------------
# 20. Save construction summary
# ------------------------------------------------------------

summary = {
    "output_file": str(OUTPUT_H5AD),
    "output_size_gb": float(output_size_gb),
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_nonzero": int(adata.X.nnz),
    "dtype": str(adata.X.dtype),
    "sparse_format": adata.X.getformat(),
    "number_of_sample_prefixes": int(
        adata.obs["sample_prefix"].nunique()
    ),
    "original_gene_symbols_unique": bool(
        adata.var["gene_symbol"].is_unique
    ),
    "anndata_var_names_unique": bool(
        adata.var_names.is_unique
    ),
    "anndata_obs_names_unique": bool(
        adata.obs_names.is_unique
    ),
    "expression_values_modified": False,
    "normalization_applied": False,
    "text_missing_value_placeholder": "unknown",
    "save_validation_passed": True,
}

with open(
    SUMMARY_FILE,
    mode="w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        summary,
        file_handle,
        indent=2,
    )

print("\n" + "=" * 72)
print("CORRECTED CELL 4 SUMMARY")
print("=" * 72)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nREADY_FOR_METADATA_INTEGRATION = True")


# ------------------------------------------------------------
# 21. Optional memory cleanup
# ------------------------------------------------------------

del expression
gc.collect()

PASS: All AnnData input files were found.

Loading sparse expression matrix...
Expression shape:    (70391, 19689)
Expression dtype:    float32
Expression nonzeros: 126,603,213
Gene metadata rows: 19,689
Cell barcodes: 70,391
PASS: Matrix dimensions match barcode and gene metadata.
PASS: Cell barcodes are unique.

Cell metadata dtypes
--------------------
sample_prefix        str
tenx_barcode         str
patient_id           str
diagnosis            str
sex                  str
cell_type            str
age              float32
dtype: object

AnnData object
--------------
AnnData object with n_obs × n_vars = 70391 × 19689
    obs: 'sample_prefix', 'tenx_barcode', 'patient_id', 'diagnosis', 'sex', 'cell_type', 'age'
    var: 'gene_symbol', 'chromosome', 'start', 'end', 'width', 'strand'
    uns: 'dataset', 'processing'
    layers: None (.X)
PASS: AnnData structure validated.

HDF5 metadata safety audit
--------------------------
sample_prefix    dtype=str          missing=0
tenx_barcode 

33

In [3]:
# ============================================================
# CELL 5: Download, parse, validate, and attach GEO metadata
# ============================================================

from pathlib import Path
import gzip
import json
import re
import shutil
import subprocess

import anndata as ad
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/Summer-Project-2026/"
    "Age_trajectory"
).resolve()

RAW_DIR = PROJECT_DIR / "data" / "raw"
METADATA_DIR = PROJECT_DIR / "data" / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "tables"
LOG_DIR = PROJECT_DIR / "logs"

for directory in [
    RAW_DIR,
    METADATA_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = (
    PROCESSED_DIR
    / "GSE200164_expression_unannotated.h5ad"
)

SERIES_MATRIX_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE200nnn/GSE200164/matrix/"
    "GSE200164_series_matrix.txt.gz"
)

SERIES_MATRIX_FILE = (
    RAW_DIR
    / "GSE200164_series_matrix.txt.gz"
)

SAMPLE_METADATA_FILE = (
    METADATA_DIR
    / "GSE200164_sample_metadata.csv"
)

CELL_METADATA_FILE = (
    METADATA_DIR
    / "GSE200164_cell_metadata.csv.gz"
)

OUTPUT_H5AD = (
    PROCESSED_DIR
    / "GSE200164_expression_with_clinical_metadata.h5ad"
)

SUMMARY_FILE = (
    LOG_DIR
    / "GSE200164_metadata_integration_summary.json"
)


# ------------------------------------------------------------
# 2. Confirm input AnnData exists
# ------------------------------------------------------------

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Input AnnData was not found:\n{INPUT_H5AD}"
    )

print(f"Input AnnData:\n{INPUT_H5AD}")


# ------------------------------------------------------------
# 3. Download GEO series matrix metadata
# ------------------------------------------------------------

curl_path = shutil.which("curl")

if curl_path is None:
    raise RuntimeError(
        "curl was not found. Install it with:\n"
        "sudo apt-get update && sudo apt-get install -y curl"
    )

if not SERIES_MATRIX_FILE.exists():
    print("\nDownloading GEO series matrix metadata...")

    command = [
        curl_path,
        "--location",
        "--fail",
        "--retry", "5",
        "--retry-delay", "5",
        "--output", str(SERIES_MATRIX_FILE),
        SERIES_MATRIX_URL,
    ]

    result = subprocess.run(
        command,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Series-matrix metadata download failed.\n"
            f"curl exit code: {result.returncode}"
        )
else:
    print(
        "\nSeries matrix already exists; download skipped."
    )

print(
    f"Series matrix file:\n{SERIES_MATRIX_FILE}"
)
print(
    f"Compressed size: "
    f"{SERIES_MATRIX_FILE.stat().st_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# 4. Read sample-level metadata lines
# ------------------------------------------------------------

metadata_rows = {}

with gzip.open(
    SERIES_MATRIX_FILE,
    mode="rt",
    encoding="utf-8",
    errors="replace",
) as handle:

    for line in handle:
        if not line.startswith("!Sample_"):
            continue

        line = line.rstrip("\n")
        fields = line.split("\t")

        field_name = fields[0].replace(
            "!Sample_",
            "",
        )

        values = [
            value.strip().strip('"')
            for value in fields[1:]
        ]

        metadata_rows.setdefault(
            field_name,
            [],
        ).append(values)


# ------------------------------------------------------------
# 5. Inspect available GEO fields
# ------------------------------------------------------------

print("\nAvailable Sample metadata fields")
print("--------------------------------")

for field_name, entries in metadata_rows.items():
    print(
        f"{field_name:35s} "
        f"rows={len(entries)} "
        f"values_in_first_row="
        f"{len(entries[0]) if entries else 0}"
    )


# ------------------------------------------------------------
# 6. Extract required aligned GEO vectors
# ------------------------------------------------------------

def require_single_metadata_row(
    field_name: str,
) -> list[str]:
    """Return a GEO sample metadata row with validation."""

    rows = metadata_rows.get(field_name)

    if rows is None:
        raise KeyError(
            f"GEO field was not found: {field_name}"
        )

    if len(rows) != 1:
        raise ValueError(
            f"Expected one row for {field_name}, "
            f"observed {len(rows)}."
        )

    return rows[0]


gsm_accessions = require_single_metadata_row(
    "geo_accession"
)

titles = require_single_metadata_row(
    "title"
)

descriptions = require_single_metadata_row(
    "description"
)

n_samples = len(gsm_accessions)

for field_name, values in [
    ("title", titles),
    ("description", descriptions),
]:
    if len(values) != n_samples:
        raise ValueError(
            f"{field_name} has {len(values)} values, "
            f"but expected {n_samples}."
        )

print(f"\nGEO samples found: {n_samples}")


# ------------------------------------------------------------
# 7. Parse repeated characteristics rows
# ------------------------------------------------------------

characteristic_rows = metadata_rows.get(
    "characteristics_ch1",
    [],
)

if not characteristic_rows:
    raise KeyError(
        "No Sample_characteristics_ch1 rows were found."
    )

for row_number, values in enumerate(
    characteristic_rows,
    start=1,
):
    if len(values) != n_samples:
        raise ValueError(
            f"Characteristics row {row_number} has "
            f"{len(values)} values; expected {n_samples}."
        )


def split_characteristic(
    text: str,
) -> tuple[str, str]:
    """
    Convert strings such as:
        'Sex: Female'
        'diagnosis: Healthy'
        'age: 67'
    into key-value pairs.
    """

    if ":" not in text:
        return text.strip().lower(), ""

    key, value = text.split(
        ":",
        maxsplit=1,
    )

    return (
        key.strip().lower(),
        value.strip(),
    )


characteristics_by_sample = [
    {}
    for _ in range(n_samples)
]

for characteristic_row in characteristic_rows:
    for sample_index, text in enumerate(
        characteristic_row
    ):
        key, value = split_characteristic(text)

        characteristics_by_sample[
            sample_index
        ][key] = value


# ------------------------------------------------------------
# 8. Build sample metadata table
# ------------------------------------------------------------

sample_records = []

for sample_index in range(n_samples):
    characteristics = characteristics_by_sample[
        sample_index
    ]

    title = titles[sample_index]
    description = descriptions[sample_index]

    # Titles look like:
    # CSF_Healthy_A1
    # CSF_MCI/AD_A6
    #
    # Description contains the concise sample prefix.
    sample_prefix = description.strip()

    # Fallback extraction if description is unexpectedly absent.
    if (
        not sample_prefix
        or sample_prefix.lower() == "unknown"
    ):
        title_match = re.search(
            r"_([A-Z]\d+)$",
            title,
        )

        if title_match is None:
            raise ValueError(
                "Could not recover sample prefix from:\n"
                f"title={title!r}\n"
                f"description={description!r}"
            )

        sample_prefix = title_match.group(1)

    sex = characteristics.get(
        "sex",
        characteristics.get(
            "gender",
            "unknown",
        ),
    )

    diagnosis = characteristics.get(
        "diagnosis",
        "unknown",
    )

    age_text = characteristics.get(
        "age",
        "",
    )

    age = pd.to_numeric(
        age_text,
        errors="coerce",
    )

    sample_records.append(
        {
            "sample_prefix": sample_prefix,
            "patient_id": sample_prefix,
            "gsm": gsm_accessions[sample_index],
            "title": title,
            "description": description,
            "sex": sex,
            "diagnosis": diagnosis,
            "age": age,
        }
    )

sample_metadata = pd.DataFrame(
    sample_records
)


# ------------------------------------------------------------
# 9. Standardize metadata values
# ------------------------------------------------------------

for column in [
    "sample_prefix",
    "patient_id",
    "gsm",
    "title",
    "description",
    "sex",
    "diagnosis",
]:
    sample_metadata[column] = (
        sample_metadata[column]
        .fillna("unknown")
        .astype(str)
        .str.strip()
    )

sample_metadata["sex"] = (
    sample_metadata["sex"]
    .str.lower()
    .replace(
        {
            "female": "Female",
            "male": "Male",
            "f": "Female",
            "m": "Male",
        }
    )
)

sample_metadata["diagnosis_original"] = (
    sample_metadata["diagnosis"]
)

diagnosis_mapping = {
    "healthy": "Healthy",
    "mci": "MCI",
    "ad": "AD",
    "alzheimer's disease": "AD",
    "alzheimers disease": "AD",
    "mci/ad": "MCI/AD",
}

sample_metadata["diagnosis"] = (
    sample_metadata["diagnosis"]
    .str.lower()
    .map(diagnosis_mapping)
    .fillna(
        sample_metadata["diagnosis_original"]
    )
)

sample_metadata["age"] = pd.to_numeric(
    sample_metadata["age"],
    errors="coerce",
).astype(float)

sample_metadata["cognitive_group"] = np.where(
    sample_metadata["diagnosis"].eq("Healthy"),
    "Healthy",
    "Cognitively impaired",
)


# ------------------------------------------------------------
# 10. Validate sample metadata
# ------------------------------------------------------------

print("\nSample metadata preview")
print("-----------------------")
display(
    sample_metadata.head(10)
)

print("\nSample metadata dimensions")
print("--------------------------")
print(sample_metadata.shape)

if n_samples != 59:
    raise AssertionError(
        f"Expected 59 GEO samples, observed {n_samples}."
    )

if not sample_metadata["sample_prefix"].is_unique:
    duplicates = (
        sample_metadata.loc[
            sample_metadata[
                "sample_prefix"
            ].duplicated(
                keep=False
            ),
            "sample_prefix",
        ]
        .tolist()
    )

    raise ValueError(
        "Sample prefixes are not unique.\n"
        f"Duplicates: {duplicates}"
    )

if not sample_metadata["gsm"].is_unique:
    raise ValueError(
        "GSM accessions are not unique."
    )

if sample_metadata["age"].isna().any():
    problematic = sample_metadata.loc[
        sample_metadata["age"].isna()
    ]

    raise ValueError(
        "One or more ages could not be parsed.\n"
        f"{problematic}"
    )

print("\nDiagnosis counts")
print("----------------")
print(
    sample_metadata["diagnosis"]
    .value_counts(dropna=False)
)

print("\nCognitive-group counts")
print("----------------------")
print(
    sample_metadata["cognitive_group"]
    .value_counts(dropna=False)
)

print("\nSex counts")
print("----------")
print(
    sample_metadata["sex"]
    .value_counts(dropna=False)
)

print("\nAge summary")
print("-----------")
print(
    sample_metadata.groupby(
        "cognitive_group"
    )["age"].describe()
)


# ------------------------------------------------------------
# 11. Validate a known GEO sample
# ------------------------------------------------------------

# GEO reports:
# A1 = GSM6016346, Female, Healthy, age 67.

a1 = sample_metadata.loc[
    sample_metadata["sample_prefix"].eq("A1")
]

if len(a1) != 1:
    raise AssertionError(
        "Expected exactly one A1 metadata record."
    )

a1_record = a1.iloc[0]

expected_a1 = {
    "gsm": "GSM6016346",
    "sex": "Female",
    "diagnosis": "Healthy",
    "age": 67.0,
}

for field, expected_value in expected_a1.items():
    observed_value = a1_record[field]

    if observed_value != expected_value:
        raise AssertionError(
            "A1 metadata validation failed.\n"
            f"Field: {field}\n"
            f"Expected: {expected_value!r}\n"
            f"Observed: {observed_value!r}"
        )

print(
    "\nPASS: A1 metadata matches the GEO record."
)


# ------------------------------------------------------------
# 12. Save sample-level metadata
# ------------------------------------------------------------

sample_metadata = sample_metadata.sort_values(
    "sample_prefix"
).reset_index(drop=True)

sample_metadata.to_csv(
    SAMPLE_METADATA_FILE,
    index=False,
)

print(
    "\nSample metadata saved to:"
    f"\n{SAMPLE_METADATA_FILE}"
)


# ------------------------------------------------------------
# 13. Load AnnData
# ------------------------------------------------------------

print("\nLoading AnnData...")

adata = ad.read_h5ad(
    INPUT_H5AD
)

print(adata)


# ------------------------------------------------------------
# 14. Compare cell prefixes to GEO sample prefixes
# ------------------------------------------------------------

cell_prefixes = set(
    adata.obs["sample_prefix"]
    .astype(str)
    .unique()
)

metadata_prefixes = set(
    sample_metadata["sample_prefix"]
    .astype(str)
    .unique()
)

missing_from_metadata = sorted(
    cell_prefixes - metadata_prefixes
)

missing_from_cells = sorted(
    metadata_prefixes - cell_prefixes
)

print("\nPrefix alignment audit")
print("----------------------")
print(
    f"Prefixes in AnnData:  {len(cell_prefixes)}"
)
print(
    f"Prefixes in metadata: {len(metadata_prefixes)}"
)
print(
    f"Missing from metadata: {missing_from_metadata}"
)
print(
    f"Missing from cells:    {missing_from_cells}"
)

if missing_from_metadata:
    raise ValueError(
        "Some cell prefixes lack GEO metadata.\n"
        f"{missing_from_metadata}"
    )

if missing_from_cells:
    raise ValueError(
        "Some GEO samples have no cells in the expression matrix.\n"
        f"{missing_from_cells}"
    )


# ------------------------------------------------------------
# 15. Attach metadata by sample prefix
# ------------------------------------------------------------

metadata_lookup = (
    sample_metadata
    .set_index("sample_prefix")
)

columns_to_attach = [
    "patient_id",
    "gsm",
    "title",
    "sex",
    "age",
    "diagnosis",
    "diagnosis_original",
    "cognitive_group",
]

for column in columns_to_attach:
    mapped_values = (
        adata.obs["sample_prefix"]
        .astype(str)
        .map(metadata_lookup[column])
    )

    if mapped_values.isna().any():
        n_missing = int(
            mapped_values.isna().sum()
        )

        raise ValueError(
            f"Metadata mapping failed for column "
            f"{column!r}: {n_missing:,} missing cells."
        )

    adata.obs[column] = mapped_values.to_numpy()


# ------------------------------------------------------------
# 16. Standardize AnnData metadata dtypes
# ------------------------------------------------------------

text_columns = [
    "sample_prefix",
    "tenx_barcode",
    "patient_id",
    "gsm",
    "title",
    "sex",
    "diagnosis",
    "diagnosis_original",
    "cognitive_group",
    "cell_type",
]

for column in text_columns:
    adata.obs[column] = (
        adata.obs[column]
        .fillna("unknown")
        .astype(str)
    )

adata.obs["age"] = pd.to_numeric(
    adata.obs["age"],
    errors="raise",
).astype(np.float32)


# ------------------------------------------------------------
# 17. Add dataset-level metadata
# ------------------------------------------------------------

adata.uns["dataset"][
    "expression_status"
] = (
    "SoupX-adjusted count-like values; fractional values are "
    "expected after ambient-RNA correction."
)

adata.uns["processing"][
    "cell_metadata_attached"
] = True

adata.uns["processing"][
    "metadata_source"
] = (
    "GSE200164 GEO series matrix"
)

adata.uns["clinical_variable_definitions"] = {
    "patient_id": (
        "Study sample identifier, equivalent to sample prefix."
    ),
    "age": "Age in years at CSF collection.",
    "sex": "Sex reported in GEO metadata.",
    "diagnosis": (
        "Healthy, MCI, AD, or combined MCI/AD label as available."
    ),
    "cognitive_group": (
        "Healthy versus cognitively impaired."
    ),
}


# ------------------------------------------------------------
# 18. Cell-level validation
# ------------------------------------------------------------

required_attached_columns = [
    "patient_id",
    "gsm",
    "age",
    "sex",
    "diagnosis",
    "cognitive_group",
]

for column in required_attached_columns:
    if adata.obs[column].isna().any():
        raise AssertionError(
            f"Missing cell metadata remains in {column}."
        )

if adata.obs["patient_id"].nunique() != 59:
    raise AssertionError(
        "Expected 59 unique patient/sample IDs, observed "
        f"{adata.obs['patient_id'].nunique()}."
    )

print("\nCell counts by cognitive group")
print("------------------------------")
print(
    adata.obs["cognitive_group"]
    .value_counts()
)

print("\nCell counts by diagnosis")
print("------------------------")
print(
    adata.obs["diagnosis"]
    .value_counts()
)

print("\nCells per patient")
print("-----------------")
cells_per_patient = (
    adata.obs["patient_id"]
    .value_counts()
)

print(
    cells_per_patient.describe()
)


# ------------------------------------------------------------
# 19. Save cell metadata separately
# ------------------------------------------------------------

adata.obs.to_csv(
    CELL_METADATA_FILE,
    compression="gzip",
)

print(
    "\nCell metadata saved to:"
    f"\n{CELL_METADATA_FILE}"
)


# ------------------------------------------------------------
# 20. Remove incomplete output if present
# ------------------------------------------------------------

if OUTPUT_H5AD.exists():
    OUTPUT_H5AD.unlink()


# ------------------------------------------------------------
# 21. Save annotated AnnData
# ------------------------------------------------------------

print(
    "\nSaving metadata-annotated AnnData to:"
    f"\n{OUTPUT_H5AD}"
)

adata.write_h5ad(
    OUTPUT_H5AD,
    compression="gzip",
    convert_strings_to_categoricals=False,
)

output_size_gb = (
    OUTPUT_H5AD.stat().st_size
    / 1024**3
)

print(
    f"Saved H5AD size: {output_size_gb:.3f} GB"
)


# ------------------------------------------------------------
# 22. Read-back validation
# ------------------------------------------------------------

print("\nReopening saved AnnData...")

adata_check = ad.read_h5ad(
    OUTPUT_H5AD,
    backed="r",
)

if adata_check.shape != adata.shape:
    raise AssertionError(
        "Saved object has an unexpected shape."
    )

if adata_check.obs_names.tolist() != adata.obs_names.tolist():
    raise AssertionError(
        "Cell ordering changed during save."
    )

for column in required_attached_columns:
    if column not in adata_check.obs.columns:
        raise AssertionError(
            f"Saved object is missing metadata column: {column}"
        )

adata_check.file.close()

print(
    "PASS: Saved annotated AnnData reopened successfully."
)


# ------------------------------------------------------------
# 23. Save integration summary
# ------------------------------------------------------------

summary = {
    "input_h5ad": str(INPUT_H5AD),
    "output_h5ad": str(OUTPUT_H5AD),
    "sample_metadata_file": str(
        SAMPLE_METADATA_FILE
    ),
    "cell_metadata_file": str(
        CELL_METADATA_FILE
    ),
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_samples": int(
        sample_metadata.shape[0]
    ),
    "n_unique_patient_ids": int(
        adata.obs["patient_id"].nunique()
    ),
    "n_healthy_samples": int(
        (
            sample_metadata["cognitive_group"]
            == "Healthy"
        ).sum()
    ),
    "n_cognitively_impaired_samples": int(
        (
            sample_metadata["cognitive_group"]
            == "Cognitively impaired"
        ).sum()
    ),
    "metadata_missing_cells": 0,
    "prefix_alignment_passed": True,
    "a1_validation_passed": True,
    "save_validation_passed": True,
}

with open(
    SUMMARY_FILE,
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )

print("\n" + "=" * 72)
print("CELL 5 SUMMARY")
print("=" * 72)

for key, value in summary.items():
    print(f"{key}: {value}")

print(
    "\nREADY_FOR_EXPRESSION_AND_QC_AUDIT = True"
)

Input AnnData:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/processed/GSE200164_expression_unannotated.h5ad

Series matrix file:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/raw/GSE200164_series_matrix.txt.gz
Compressed size: 0.00 MB

Available Sample metadata fields
--------------------------------
title                               rows=1 values_in_first_row=59
geo_accession                       rows=1 values_in_first_row=59
status                              rows=1 values_in_first_row=59
submission_date                     rows=1 values_in_first_row=59
last_update_date                    rows=1 values_in_first_row=59
type                                rows=1 values_in_first_row=59
channel_count                       rows=1 values_in_first_row=59
source_name_ch1                     rows=1 values_in_first_row=59
organism_ch1                        rows=1 values_in_first_row=59
characteristics_ch1      

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4541  100  4541    0     0  49158      0 --:--:-- --:--:-- --:--:-- 49358


,sample_prefix,patient_id,gsm,title,description,sex,diagnosis,age,diagnosis_original,cognitive_group
0,A1,A1,GSM6016346,CSF_Healthy_A1,A1,Female,Healthy,67.0,Healthy,Healthy
1,A2,A2,GSM6016347,CSF_Healthy_A2,A2,Male,Healthy,77.0,Healthy,Healthy
2,A4,A4,GSM6016348,CSF_Healthy_A4,A4,Male,Healthy,67.0,Healthy,Healthy
3,A5,A5,GSM6016349,CSF_Healthy_A5,A5,Male,Healthy,72.0,Healthy,Healthy
4,A6,A6,GSM6016350,CSF_MCI/AD_A6,A6,Male,MCI/AD,78.0,MCI/AD,Cognitively impaired
5,A7,A7,GSM6016351,CSF_Healthy_A7,A7,Female,Healthy,63.0,Healthy,Healthy
6,A8,A8,GSM6016352,CSF_Healthy_A8,A8,Female,Healthy,71.0,Healthy,Healthy
7,B1,B1,GSM6016353,CSF_MCI/AD_B1,B1,Male,MCI/AD,70.0,MCI/AD,Cognitively impaired
8,B2,B2,GSM6016354,CSF_Healthy_B2,B2,Female,Healthy,67.0,Healthy,Healthy
9,B3,B3,GSM6016355,CSF_Healthy_B3,B3,Male,Healthy,71.0,Healthy,Healthy



Sample metadata dimensions
--------------------------
(59, 10)

Diagnosis counts
----------------
diagnosis
Healthy    45
MCI/AD     14
Name: count, dtype: int64

Cognitive-group counts
----------------------
cognitive_group
Healthy                 45
Cognitively impaired    14
Name: count, dtype: int64

Sex counts
----------
sex
Female    34
Male      25
Name: count, dtype: int64

Age summary
-----------
                      count       mean       std   min   25%   50%    75%  \
cognitive_group                                                             
Cognitively impaired   14.0  69.500000  9.354143  48.0  64.0  72.5  76.75   
Healthy                45.0  68.955556  6.574591  54.0  65.0  69.0  73.00   

                       max  
cognitive_group             
Cognitively impaired  79.0  
Healthy               82.0  

PASS: A1 metadata matches the GEO record.

Sample metadata saved to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/metadata/G

In [8]:
sample_metadata_raw = sample_metadata.copy()

sample_metadata = sample_metadata[
    [
        "patient_id",
        "gsm",
        "title",
        "sex",
        "age",
        "diagnosis_original",
        "cognitive_group",
    ]
].copy()

In [10]:
# ============================================================
# CELL 5A: Audit sample metadata before attaching it to AnnData
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Inspect the cleaned metadata
# ------------------------------------------------------------

print("Sample metadata shape:", sample_metadata.shape)

print("\nColumns:")
print(sample_metadata.columns.tolist())

print("\nFirst rows:")
display(sample_metadata.head(10))


# ------------------------------------------------------------
# 2. Normalize string fields
# ------------------------------------------------------------

string_columns = [
    "patient_id",
    "gsm",
    "title",
    "sex",
    "diagnosis_original",
    "cognitive_group",
]

for column in string_columns:
    sample_metadata[column] = (
        sample_metadata[column]
        .astype("string")
        .str.strip()
    )

sample_metadata["age"] = pd.to_numeric(
    sample_metadata["age"],
    errors="coerce",
)


# ------------------------------------------------------------
# 3. Inspect unique counts
# ------------------------------------------------------------

print("\nUnique-value counts")
print("-------------------")

for column in sample_metadata.columns:
    print(
        f"{column:22s}: "
        f"{sample_metadata[column].nunique(dropna=True):,}"
    )


# ------------------------------------------------------------
# 4. Check missing values
# ------------------------------------------------------------

missing_summary = (
    sample_metadata
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

missing_summary["fraction_missing"] = (
    missing_summary["missing_values"]
    / len(sample_metadata)
)

print("\nMissing-value summary:")
display(missing_summary)


# ------------------------------------------------------------
# 5. Check whether each identifier is unique
# ------------------------------------------------------------

identifier_columns = [
    "patient_id",
    "gsm",
    "title",
]

print("\nIdentifier uniqueness")
print("---------------------")

for column in identifier_columns:
    duplicated = sample_metadata[column].duplicated(
        keep=False
    )

    print(
        f"{column:12s}: "
        f"unique={sample_metadata[column].is_unique}, "
        f"duplicated rows={duplicated.sum():,}"
    )

    if duplicated.any():
        display(
            sample_metadata.loc[
                duplicated,
                [
                    "patient_id",
                    "gsm",
                    "title",
                    "age",
                    "sex",
                    "cognitive_group",
                ],
            ]
            .sort_values(column)
            .head(30)
        )


# ------------------------------------------------------------
# 6. Inspect AnnData barcode prefixes
# ------------------------------------------------------------

prefix_table = (
    adata.obs["sample_prefix"]
    .value_counts()
    .rename_axis("sample_prefix")
    .reset_index(name="n_cells")
    .sort_values("sample_prefix")
)

print("\nAnnData sample prefixes:")
print("Number of prefixes:", len(prefix_table))

display(prefix_table.head(30))


# ------------------------------------------------------------
# 7. Test direct overlap with metadata fields
# ------------------------------------------------------------

adata_prefixes = set(
    adata.obs["sample_prefix"]
    .dropna()
    .astype(str)
)

overlap_records = []

for column in identifier_columns:
    metadata_values = set(
        sample_metadata[column]
        .dropna()
        .astype(str)
    )

    overlap = adata_prefixes & metadata_values

    overlap_records.append(
        {
            "metadata_column": column,
            "n_adata_prefixes": len(adata_prefixes),
            "n_metadata_values": len(metadata_values),
            "n_overlapping_values": len(overlap),
            "fraction_of_prefixes_matched": (
                len(overlap) / len(adata_prefixes)
                if len(adata_prefixes) > 0
                else np.nan
            ),
            "example_matches": sorted(overlap)[:10],
        }
    )

overlap_table = pd.DataFrame(overlap_records)

print("\nDirect prefix-to-metadata overlap:")
display(overlap_table)


# ------------------------------------------------------------
# 8. Show unmatched values for the best candidate
# ------------------------------------------------------------

best_candidate = (
    overlap_table
    .sort_values(
        "n_overlapping_values",
        ascending=False,
    )
    .iloc[0]
)

best_column = best_candidate["metadata_column"]

metadata_identifier_values = set(
    sample_metadata[best_column]
    .dropna()
    .astype(str)
)

unmatched_prefixes = sorted(
    adata_prefixes - metadata_identifier_values
)

unused_metadata_values = sorted(
    metadata_identifier_values - adata_prefixes
)

print("\nBest direct candidate:", best_column)

print(
    f"Matched prefixes: "
    f"{best_candidate['n_overlapping_values']}/"
    f"{best_candidate['n_adata_prefixes']}"
)

print("\nUnmatched AnnData prefixes:")
print(unmatched_prefixes[:50])

print("\nMetadata identifiers not present in AnnData:")
print(unused_metadata_values[:50])

Sample metadata shape: (59, 7)

Columns:
['patient_id', 'gsm', 'title', 'sex', 'age', 'diagnosis_original', 'cognitive_group']

First rows:


,patient_id,gsm,title,sex,age,diagnosis_original,cognitive_group
0,A1,GSM6016346,CSF_Healthy_A1,Female,67.0,Healthy,Healthy
1,A2,GSM6016347,CSF_Healthy_A2,Male,77.0,Healthy,Healthy
2,A4,GSM6016348,CSF_Healthy_A4,Male,67.0,Healthy,Healthy
3,A5,GSM6016349,CSF_Healthy_A5,Male,72.0,Healthy,Healthy
4,A6,GSM6016350,CSF_MCI/AD_A6,Male,78.0,MCI/AD,Cognitively impaired
5,A7,GSM6016351,CSF_Healthy_A7,Female,63.0,Healthy,Healthy
6,A8,GSM6016352,CSF_Healthy_A8,Female,71.0,Healthy,Healthy
7,B1,GSM6016353,CSF_MCI/AD_B1,Male,70.0,MCI/AD,Cognitively impaired
8,B2,GSM6016354,CSF_Healthy_B2,Female,67.0,Healthy,Healthy
9,B3,GSM6016355,CSF_Healthy_B3,Male,71.0,Healthy,Healthy



Unique-value counts
-------------------
patient_id            : 59
gsm                   : 59
title                 : 59
sex                   : 2
age                   : 24
diagnosis_original    : 2
cognitive_group       : 2

Missing-value summary:


,missing_values,fraction_missing
patient_id,0,0.0
gsm,0,0.0
title,0,0.0
sex,0,0.0
age,0,0.0
diagnosis_original,0,0.0
cognitive_group,0,0.0



Identifier uniqueness
---------------------
patient_id  : unique=True, duplicated rows=0
gsm         : unique=True, duplicated rows=0
title       : unique=True, duplicated rows=0

AnnData sample prefixes:
Number of prefixes: 59


,sample_prefix,n_cells
35,A1,684
1,A2,4169
11,A4,1682
27,A5,1001
18,A6,1316
21,A7,1239
58,A8,151
29,B1,966
31,B2,843
36,B3,668



Direct prefix-to-metadata overlap:


,metadata_column,n_adata_prefixes,n_metadata_values,n_overlapping_values,fraction_of_prefixes_matched,example_matches
0,patient_id,59,59,59,1.0,"[A1, A2, A4, A5, A6, A7, A8, B1, B2, B3]"
1,gsm,59,59,0,0.0,[]
2,title,59,59,0,0.0,[]



Best direct candidate: patient_id
Matched prefixes: 59/59

Unmatched AnnData prefixes:
[]

Metadata identifiers not present in AnnData:
[]


In [12]:
adata_prefixes = set(
    adata.obs["sample_prefix"].astype(str)
)

patient_ids = set(
    sample_metadata["patient_id"].astype(str)
)

print("AnnData prefixes:", len(adata_prefixes))
print("Metadata patient IDs:", len(patient_ids))
print("Overlap:", len(adata_prefixes & patient_ids))

print("\nPrefixes not matched to patient IDs:")
print(sorted(adata_prefixes - patient_ids)[:50])

print("\nPatient IDs not found among prefixes:")
print(sorted(patient_ids - adata_prefixes)[:50])

AnnData prefixes: 59
Metadata patient IDs: 59
Overlap: 59

Prefixes not matched to patient IDs:
[]

Patient IDs not found among prefixes:
[]


In [13]:
# ============================================================
# CELL 5B: Attach patient metadata to each cell
# ============================================================

import numpy as np
import pandas as pd

metadata_by_patient = (
    sample_metadata
    .drop_duplicates("patient_id")
    .set_index("patient_id")
)

if not metadata_by_patient.index.is_unique:
    raise ValueError("patient_id is not unique.")

cell_patient_ids = adata.obs["sample_prefix"].astype(str)

unmatched = sorted(
    set(cell_patient_ids)
    - set(metadata_by_patient.index.astype(str))
)

if unmatched:
    raise ValueError(
        f"Unmatched cell prefixes: {unmatched[:20]}"
    )

adata.obs["patient_id"] = cell_patient_ids.values

for column in [
    "gsm",
    "title",
    "sex",
    "age",
    "diagnosis_original",
    "cognitive_group",
]:
    adata.obs[column] = (
        cell_patient_ids
        .map(metadata_by_patient[column])
        .values
    )

# Validate no metadata were lost.
for column in [
    "patient_id",
    "age",
    "sex",
    "diagnosis_original",
    "cognitive_group",
]:
    missing = int(adata.obs[column].isna().sum())
    print(f"{column:22s}: {missing:,} missing")

    if missing:
        raise ValueError(
            f"Missing values after merge in {column}"
        )

# Use compact types.
adata.obs["age"] = pd.to_numeric(
    adata.obs["age"],
    errors="raise",
).astype(np.float32)

for column in [
    "patient_id",
    "gsm",
    "title",
    "sex",
    "diagnosis_original",
    "cognitive_group",
]:
    adata.obs[column] = adata.obs[column].astype("category")

print("\nMetadata attachment complete.")
print(adata)

patient_id            : 0 missing
age                   : 0 missing
sex                   : 0 missing
diagnosis_original    : 0 missing
cognitive_group       : 0 missing

Metadata attachment complete.
AnnData object with n_obs × n_vars = 70391 × 19689
    obs: 'sample_prefix', 'tenx_barcode', 'patient_id', 'diagnosis', 'sex', 'cell_type', 'age', 'gsm', 'title', 'diagnosis_original', 'cognitive_group'
    var: 'gene_symbol', 'chromosome', 'start', 'end', 'width', 'strand'
    uns: 'dataset', 'processing', 'clinical_variable_definitions'
    layers: None (.X)


In [14]:
patient_table = (
    adata.obs[
        [
            "patient_id",
            "gsm",
            "title",
            "age",
            "sex",
            "diagnosis_original",
            "cognitive_group",
        ]
    ]
    .drop_duplicates()
    .sort_values("patient_id")
)

print("Number of patients:", patient_table.shape[0])

print("\nPatients by cognitive group:")
display(
    patient_table["cognitive_group"]
    .value_counts(dropna=False)
)

print("\nSex by cognitive group:")
display(
    pd.crosstab(
        patient_table["cognitive_group"],
        patient_table["sex"],
        margins=True,
    )
)

print("\nAge summary by cognitive group:")
display(
    patient_table
    .groupby("cognitive_group", observed=True)["age"]
    .describe()
)

print("\nCells per patient:")
display(
    adata.obs["patient_id"]
    .value_counts()
    .describe()
)

Number of patients: 59

Patients by cognitive group:


cognitive_group
Healthy                 45
Cognitively impaired    14
Name: count, dtype: int64


Sex by cognitive group:


sex,Female,Male,All
cognitive_group,,,
Cognitively impaired,7,7,14
Healthy,27,18,45
All,34,25,59



Age summary by cognitive group:


,count,mean,std,min,25%,50%,75%,max
cognitive_group,,,,,,,,
Cognitively impaired,14.0,69.500000,9.354143,48.0,64.0,72.5,76.75,79.0
Healthy,45.0,68.955559,6.574592,54.0,65.0,69.0,73.00,82.0



Cells per patient:


count      59.000000
mean     1193.067797
std       946.411131
min       151.000000
25%       569.000000
50%       966.000000
75%      1441.500000
max      4645.000000
Name: count, dtype: float64

In [ ]:
adata.write_h5ad(
    PROCESSED_DIR /
    "GSE200164_canonical_input.h5ad",
    compression="gzip",
)